# Phase 4 — Customer Segmentation (K-Means Clustering)

**Goal:** Automatically group customers into behavioral segments using an unsupervised machine learning algorithm called K-Means.

In Phase 3 we created rule-based segments (Champions, At Risk, etc.) using business logic. Here we let the **data itself** find the natural groupings — no human-defined rules, just math.

This is an **unsupervised learning** problem: we have no labels (no 'correct' answer). The algorithm finds structure in the data on its own.

**Business use case:** Personalised marketing — send discount coupons only to 'At Risk' customers, give loyalty rewards to 'Champions', re-engage 'Lost' customers with a campaign.

## 0. Setup & Load Data

We load the `customer_features.csv` produced in Phase 3. This file has one row per customer with their R, F, M values and extra features.

We also import `scikit-learn` — Python's standard machine learning library. Key classes:
- `StandardScaler`: scales features to zero mean, unit variance (required before K-Means)
- `KMeans`: the clustering algorithm
- `PCA`: Principal Component Analysis — reduces dimensions so we can visualize clusters in 2D
- `silhouette_score`: measures cluster quality (1 = perfect, -1 = wrong clusters)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.titlesize': 14})

# Load customer features from Phase 3
df = pd.read_csv('../data/customer_features.csv')
print(f'Loaded {len(df):,} customers')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Select & Prepare Features for Clustering

K-Means works on **numerical distances** between data points. This means:
1. We can only use numerical columns
2. All features must be on the **same scale** — otherwise a feature with large values (like `monetary_value` in BRL) will dominate the distance calculation over a feature with small values (like `frequency`)

**StandardScaler** transforms each feature so that:
- Mean = 0
- Standard deviation = 1

Formula: `z = (x - mean) / std`

We use the core RFM features: `recency_days`, `frequency`, `monetary_value`, `avg_order_value`.

In [ ]:
# Select the features we'll cluster on
feature_cols = ['recency_days', 'frequency', 'monetary_value', 'avg_order_value']

# Drop rows with any nulls in these columns
df_clean = df[['customer_unique_id'] + feature_cols].dropna()
print(f'Rows after dropping nulls: {len(df_clean):,}')

# Extract the feature matrix (X)
X = df_clean[feature_cols].values

# Scale features: fit_transform computes mean & std, then applies z-score transformation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check the scaling worked: mean should be ~0, std should be ~1
print('\nAfter scaling:')
print(f'  Mean per feature:  {X_scaled.mean(axis=0).round(4)}')
print(f'  Std  per feature:  {X_scaled.std(axis=0).round(4)}')

## 2. The Elbow Method — Finding the Optimal Number of Clusters (K)

K-Means requires you to tell it **how many clusters (K) to create**. But we don't know the right K upfront!

**The Elbow Method:** Train K-Means for K = 2, 3, 4, ..., 10. For each K, record the **inertia** (sum of squared distances from each point to its cluster center — a.k.a. WCSS: Within-Cluster Sum of Squares).

As K increases, inertia always decreases (more clusters = smaller groups = closer to center). But at some K, the improvement starts to level off — this is the **elbow point**. Beyond this K, adding more clusters gives diminishing returns.

**We pick K at the 'elbow'.**

In [ ]:
# Run K-Means for K = 2 to 10 and record inertia + silhouette scores
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=5000, random_state=42)
    silhouette_scores.append(sil)
    print(f'  K={k}: Inertia={km.inertia_:.0f}, Silhouette={sil:.4f}')

# Plot Elbow + Silhouette side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — Find the Bend')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — Higher is Better')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Choosing Optimal K for K-Means', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

best_k_sil = list(k_range)[silhouette_scores.index(max(silhouette_scores))]
print(f'\nBest K by Silhouette Score: {best_k_sil}')

## 3. Train the Final K-Means Model

We'll use **K=4** as our final model — it typically gives a good elbow for Olist data and produces 4 interpretable business segments.

**K-Means++ initialization** (`init='k-means++'`): Instead of placing cluster centers randomly (which can give bad results), K-Means++ spreads the initial centers out intelligently. This leads to faster convergence and better results.

**`n_init=10`**: The algorithm runs 10 times with different random starting points and picks the best result. This prevents getting stuck in a local minimum.

In [ ]:
# Train final model with K=4
OPTIMAL_K = 4

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',    # smart initialization
    n_init=10,           # run 10 times, pick best
    random_state=42      # reproducibility
)
kmeans.fit(X_scaled)

# Assign cluster labels back to customers
df_clean = df_clean.copy()
df_clean['cluster'] = kmeans.labels_

# Cluster sizes
cluster_counts = df_clean['cluster'].value_counts().sort_index()
print('Cluster sizes:')
for c, count in cluster_counts.items():
    pct = count / len(df_clean) * 100
    print(f'  Cluster {c}: {count:,} customers ({pct:.1f}%)')

print(f'\nFinal Inertia: {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, kmeans.labels_):.4f}')

## 4. Interpret Each Cluster (Cluster Profiling)

Raw cluster numbers (0, 1, 2, 3) mean nothing. We need to **look at the mean values** of each feature per cluster to understand what each cluster represents.

This is called **cluster profiling** — the most important step after clustering. Without it, you just have numbered groups with no business meaning.

**Interpretation guide:**
- Low `recency_days` + High `frequency` + High `monetary` → **Champions**
- High `recency_days` + Low `frequency` → **Lost / At-Risk**
- Low `recency_days` + Low `frequency` → **New Customers**

In [ ]:
# Cluster profiles: mean of each feature per cluster
cluster_profile = df_clean.groupby('cluster')[feature_cols].mean().round(2)
print('Cluster Profiles (mean values per cluster):')
print(cluster_profile.to_string())

# Assign human-readable labels based on the profile
# (These may need adjustment based on what your data shows)
def label_cluster(row, profiles):
    """Label clusters based on their relative R, F, M positions."""
    # Find rank of this cluster on each dimension
    r_rank = profiles['recency_days'].rank()[row.name]   # lower recency = more recent = better
    f_rank = profiles['frequency'].rank()[row.name]      # higher freq = better
    m_rank = profiles['monetary_value'].rank()[row.name] # higher monetary = better
    
    score = (5 - r_rank) + f_rank + m_rank  # inverted recency
    return score

scores = cluster_profile.apply(label_cluster, axis=1, profiles=cluster_profile)
score_sorted = scores.sort_values(ascending=False)
label_map = {}
labels = ['Champions', 'Loyal Customers', 'At-Risk', 'Lost / Inactive']
for i, (cluster_id, _) in enumerate(score_sorted.items()):
    label_map[cluster_id] = labels[i]

print('\nCluster → Segment Label mapping:')
for k, v in label_map.items():
    print(f'  Cluster {k} → {v}')

df_clean['segment'] = df_clean['cluster'].map(label_map)

## 5. Visualize Clusters with PCA

We have 4 features — impossible to plot in 4D. **PCA (Principal Component Analysis)** reduces dimensions while preserving as much variance as possible.

PCA finds the **principal components** — new axes that capture the most variance in the data. We reduce to 2 principal components (PC1, PC2) so we can make a 2D scatter plot.

**Important:** PCA is used here **only for visualization** — the actual clustering was done on all 4 original features.

In [ ]:
# Reduce to 2D with PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA Explained Variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total variance captured by 2 components: {sum(pca.explained_variance_ratio_)*100:.1f}%')

# Create a DataFrame for plotting
df_pca = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1], 
                        'cluster': df_clean['cluster'].values,
                        'segment': df_clean['segment'].values})

# Color palette for segments
palette = {
    'Champions':       '#2ecc71',
    'Loyal Customers': '#3498db',
    'At-Risk':         '#e67e22',
    'Lost / Inactive': '#e74c3c'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA scatter plot
for segment, color in palette.items():
    mask = df_pca['segment'] == segment
    axes[0].scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                   c=color, label=segment, alpha=0.4, s=5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].set_title('Customer Clusters in PCA Space')
axes[0].legend(markerscale=3)

# Segment size bar chart
seg_counts = df_clean['segment'].value_counts()
colors = [palette.get(s, 'gray') for s in seg_counts.index]
bars = axes[1].bar(seg_counts.index, seg_counts.values, color=colors, edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, seg_counts.values):
    pct = count / len(df_clean) * 100
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Customer Count per Segment')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('K-Means Customer Segmentation Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Cluster Profile Heatmap

A heatmap of cluster profiles is the standard way to present segmentation results to stakeholders. It shows at a glance which segment is high/low on each feature.

We normalize each column (z-score) so that features with different scales are comparable in the heatmap.

In [ ]:
# Profile heatmap with segment labels as index
profile_labeled = df_clean.groupby('segment')[feature_cols].mean()

# Normalize for heatmap (z-score per column)
profile_norm = (profile_labeled - profile_labeled.mean()) / profile_labeled.std()

plt.figure(figsize=(10, 5))
sns.heatmap(profile_norm, 
            annot=profile_labeled.round(1),  # show actual values
            fmt='g',
            cmap='RdYlGn',
            linewidths=0.5,
            cbar_kws={'label': 'Normalized Value (z-score)'})
plt.title('Cluster Profile Heatmap\n(Color = z-score, Numbers = actual mean value)', fontsize=14)
plt.ylabel('Customer Segment')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('\nCluster Profile (actual mean values):')
print(profile_labeled.round(2).to_string())

## 7. Business Recommendations per Segment

In a real job, the goal of clustering isn't just to have nice charts — it's to drive **action**. For each segment, you'd recommend a specific marketing or retention strategy.

In [ ]:
recommendations = {
    'Champions': {
        'action': 'Reward & Upsell',
        'tactic': 'Loyalty program, early access to new products, referral incentives',
        'priority': 'HIGH'
    },
    'Loyal Customers': {
        'action': 'Nurture & Retain',
        'tactic': 'Personalized recommendations, birthday discounts, subscription offers',
        'priority': 'HIGH'
    },
    'At-Risk': {
        'action': 'Re-engage',
        'tactic': 'Win-back email campaign, 20% off coupon, survey to understand churn reason',
        'priority': 'MEDIUM'
    },
    'Lost / Inactive': {
        'action': 'Last attempt or write-off',
        'tactic': 'Single re-engagement push, suppress from regular campaigns to save cost',
        'priority': 'LOW'
    }
}

print('=' * 65)
print('BUSINESS RECOMMENDATIONS PER SEGMENT')
print('=' * 65)
for seg, info in recommendations.items():
    count = (df_clean['segment'] == seg).sum()
    pct = count / len(df_clean) * 100
    print(f'\n[{info["priority"]}] {seg} ({count:,} customers, {pct:.1f}%)')
    print(f'  Action : {info["action"]}')
    print(f'  Tactics: {info["tactic"]}')

## 8. Save Segmentation Results

In [ ]:
# Merge segment back to full features dataframe
df_segmented = df.merge(df_clean[['customer_unique_id', 'cluster', 'segment']], 
                         on='customer_unique_id', how='left')

output_path = '../data/customer_segments.csv'
df_segmented.to_csv(output_path, index=False)
print(f'Saved {len(df_segmented):,} customer records with segments to {output_path}')
print(f'\nSegment distribution:')
print(df_segmented['segment'].value_counts())

## 9. Interview Talking Points

Here's how to explain this notebook in an interview:

1. **Why K-Means?** It's simple, scalable, and interpretable — perfect for customer segmentation where you need to explain clusters to business stakeholders. Alternatives like DBSCAN handle arbitrary shapes but are harder to explain.

2. **Why scale features?** K-Means uses Euclidean distance. Without scaling, `monetary_value` (in hundreds of BRL) would dominate `frequency` (1-5). StandardScaler brings all features to the same scale.

3. **Why K-Means++?** Random initialization can get stuck in poor local minima. K-Means++ initializes centers spread apart, consistently giving better results.

4. **How did you choose K=4?** Combined the Elbow Method (look for the bend in inertia curve) and Silhouette Score (higher = better-separated clusters). Also considered business interpretability — 4 segments map cleanly to real marketing strategies.

5. **What is PCA used for here?** Only for visualization — to project 4D feature space into 2D for the scatter plot. The clustering itself was done on all 4 original scaled features.

6. **What is Silhouette Score?** Ranges from -1 to 1. For each point: `(b - a) / max(a, b)` where `a` = average distance to same cluster, `b` = average distance to nearest other cluster. Higher = tighter clusters, better separation.